# ⚛️ RATISS-GI — ENTRAÎNEMENT COLOSSAL ULTIME : 10+ Go VRAM & 30M TOKENS/SEC

**Jonathan Evina · RATISS Labs · Yaoundé, Cameroun 🇨🇲**

> **🚀 EXPLOITATION MAXIMALE GPU VRAM (10 Go+ sur 15 Go) :**  
> - Réservation d'un buffer de travail VRAM de **10 Go** pour saturer la mémoire vidéo de la Tesla T4.
> - **3 CUDA Streams Parallèles Asynchrones** exécutant en simultané les Bigrammes, Trigrammes (60-bit pack) et Thèmes.
> - Débit visé : **10 000 000 à 30 000 000+ tokens / seconde** sans aucun Transformer !

> **🔄 REPRISE INCRÉMENTALE CONTINUE :**  
> - Chargement direct du modèle souverain de **22,9 Mo** (`ratiss_self/bouche.bouche`, commit `1978a2d`, 881 690 arcs et 1 967 213 trigrammes).
> - Les nouvelles données massives viennent s'additionner et fusionner en continu sans jamais repartir de zéro.

> **📚 LES 2 CORPUS LES PLUS LOURDS DE FRANCE (1 441 447 DIALOGUES) :**  
> 1. **Open-Hermes-FR (`legmlai/openhermes-fr`)** : Les 3 parties complètes (799 875 paires, distillation GPT-4o, 1,4 Go).
> 2. **Compar:IA Conversations (`Data-Gouv-ML/compar-ia`)** : L'intégrale (641 572 tours réels multi-tours du Ministère de la Culture).

## 1. Détection VRAM 15 Go & Chargement du Modèle Scellé (22,9 Mo)

In [ ]:
import os, subprocess, sys, shutil, torch
from pathlib import Path

print('🚀 DÉTECTION DU MATÉRIEL GPU & VRAM :')
if torch.cuda.is_available():
    gpu_nom = torch.cuda.get_device_name(0)
    vram_totale = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'  ✓ GPU Actif : {gpu_nom}')
    print(f'  ✓ VRAM Dédiée Totale : {vram_totale:.2f} Go')
else:
    print('  ⚠️ Aucun GPU détecté : basculement sur CPU vectorisé.')
    print('  💡 Conseil : Menu Exécution > Modifier le type d\'exécution > T4 GPU.')

# Clonage ou mise à jour directe depuis GitHub
if not os.path.exists('ratiss_v3.py'):
    if os.path.exists('Ratiss-GI/ratiss_v3.py'):
        os.chdir('Ratiss-GI')
    else:
        subprocess.check_call(['git', 'clone', 'https://github.com/jonathansearch/Ratiss-GI.git'])
        os.chdir('Ratiss-GI')

# Forcer la synchronisation avec le commit 1978a2d
print('🔄 Synchronisation avec le dépôt GitHub (récupération de ratiss_self)...')
subprocess.run(['git', 'pull', 'origin', 'main'])

# Détection du modèle officiel de 22,9 Mo
CANDIDATS = [
    'ratiss_self/bouche.bouche',
    'sortie/langue_massive/bouche.bouche',
    'bouche.bouche'
]
MODELE_A_CHARGER = None
for c in CANDIDATS:
    if os.path.exists(c) and os.path.getsize(c) > 1000:
        MODELE_A_CHARGER = c
        break

if MODELE_A_CHARGER:
    taille_mo = os.path.getsize(MODELE_A_CHARGER) / 1e6
    print(f'\n📦 MODÈLE SOUVERAIN CHARGÉ : {MODELE_A_CHARGER} ({taille_mo:.2f} Mo)')
    print('  ✓ Ce modèle sera gonflé en continu sur le GPU VRAM sans repartir de zéro !')
else:
    print('\n🌱 Aucun modèle antérieur trouvé : initialisation vierge.')

# Exécution des 27 tests unitaires de conformité
!python3 -m unittest discover -s tests -p 'test_*.py'

## 2. Démonstration du Pilier 4 : Résonance d'Onde & Normalisation par le Degré

Vérification de l'étouffement des hubs syntaxiques (`de`, `le`) par la loi $\frac{\text{poids}}{\ln(\text{degré})}$ et émergence de `lune` au **Rang #2**.

In [ ]:
from ratiss_v3 import TissuScelle

tissu = TissuScelle('cerveau/RatissOne.ratiss')
res = tissu.raisonner_par_resonance(['soleil', 'maree'], seuil_energie=0.01, max_sauts=2, normaliser_degre=True)

print('🏆 Top 5 des carrefours émergents sur le tissu scellé :')
for i, c in enumerate(res['carrefours'][:5]):
    print(f"  #{i+1} {c['noeud']:<15} énergie: {c['energie']:.4f} (sauts: {c['sauts']})")

## 3. Ingestion Haute Vitesse : Open-Hermes-FR (799k) + Compar:IA (641k)

Téléchargement direct et parsing multi-threadé des 4 fichiers Parquet souverains vérifiés (200 OK) :
1. `OpenHermes-FR` Partie 0 (253 Mo)
2. `OpenHermes-FR` Partie 1 (252 Mo)
3. `OpenHermes-FR` Partie 2 (253 Mo)
4. `Compar:IA Conversations` (132 Mo)

*`LIMITE_PAR_SOURCE` peut être mis à `None` pour l'intégralité des 1,44 million d'échanges, ou à une valeur (ex: 100 000) pour aller encore plus vite.*

In [ ]:
!pip install -q pyarrow fastparquet

import os, urllib.request, pandas as pd
from pathlib import Path

corpus_dir = Path('corpus')
corpus_dir.mkdir(exist_ok=True)
lignes_dialogues = []
headers = {'User-Agent': 'Mozilla/5.0'}

# Nombre d'exemples par source (None = TOUT LE CORPUS DE 1,44 MILLION !)
LIMITE_PAR_SOURCE = 150000

print('========================================================================')
print('📥 TÉLÉCHARGEMENT DIRECT DES 4 PARQUETS SOUVERAINS (VÉRIFIÉS 200 OK)')
print('========================================================================')

# 0. Ingestion préalable des acquis de ratiss_self/
self_edu = Path('ratiss_self/01_education.txt')
if self_edu.exists():
    lignes_self = [l.strip() for l in self_edu.read_text(encoding='utf-8').splitlines() if len(l.strip()) > 3]
    lignes_dialogues.extend(lignes_self)
    print(f'\n0️⃣ Acquis ratiss_self/ indexés : {len(lignes_self):,} lignes ajoutées.')

# 1. Ingestion de Open-Hermes-FR (3 Parties, 799 875 paires)
PARQUETS_HERMES = [
    ('OpenHermes Partie 0', 'https://huggingface.co/datasets/legmlai/openhermes-fr/resolve/main/data/train-00000-of-00003.parquet'),
    ('OpenHermes Partie 1', 'https://huggingface.co/datasets/legmlai/openhermes-fr/resolve/main/data/train-00001-of-00003.parquet'),
    ('OpenHermes Partie 2', 'https://huggingface.co/datasets/legmlai/openhermes-fr/resolve/main/data/train-00002-of-00003.parquet'),
]

print('\n📚 1️⃣ Ingestion de Open-Hermes-FR (Distillation GPT-4o)...')
for nom, url in PARQUETS_HERMES:
    try:
        df = pd.read_parquet(url, columns=['prompt', 'accepted_completion'])
        if LIMITE_PAR_SOURCE:
            df = df.head(LIMITE_PAR_SOURCE // 3)
        c = 0
        for _, row in df.iterrows():
            q = str(row['prompt']).strip()
            r = str(row['accepted_completion']).strip()
            if q and r:
                q_c = ' '.join(q.split())[:300]
                r_c = ' '.join(r.split())[:450]
                lignes_dialogues.append(f"_q_ {q_c} _r_ {r_c}")
                c += 1
        print(f'   ✓ {nom} : +{c:,} paires ajoutées.')
    except Exception as e:
        print(f'   ⚠️ {nom} : {e}')

# 2. Ingestion de Compar:IA Conversations (641 572 tours réels)
print('\n💬 2️⃣ Ingestion de Compar:IA Conversations (Ministère de la Culture)...')
url_comparia = 'https://huggingface.co/datasets/Data-Gouv-ML/compar-ia/resolve/main/data/comparia-conversations.parquet'
try:
    df_c = pd.read_parquet(url_comparia)
    if LIMITE_PAR_SOURCE:
        df_c = df_c.head(LIMITE_PAR_SOURCE)
    c_comp = 0
    for _, row in df_c.iterrows():
        conv = row.get('full_conversation_a') if 'full_conversation_a' in row else None
        if conv is not None and isinstance(conv, (list, tuple)):
            tours_txt = []
            for turn in conv:
                if isinstance(turn, dict):
                    role = turn.get('role', '')
                    content = ' '.join(str(turn.get('content', '')).split())[:350]
                    if role == 'user':
                        tours_txt.append(f"_q_ {content}")
                    elif role == 'assistant':
                        tours_txt.append(f"_r_ {content}")
            if tours_txt:
                lignes_dialogues.append(' '.join(tours_txt))
                c_comp += 1
        elif 'prompt' in row and 'response' in row:
            q = ' '.join(str(row['prompt']).split())[:300]
            r = ' '.join(str(row['response']).split())[:400]
            lignes_dialogues.append(f"_q_ {q} _r_ {r}")
            c_comp += 1
    print(f'   ✓ Compar:IA : +{c_comp:,} dialogues multi-tours ajoutés.')
except Exception as e:
    print(f'   ⚠️ Compar:IA : {e}')

# Consolidation du méga-corpus
fichier_corpus = corpus_dir / 'corpus_colossal.txt'
fichier_corpus.write_text('\n'.join(lignes_dialogues), encoding='utf-8')
print('\n' + '='*70)
print(f'✅ MÉGA-CORPUS COLOSSAL CONSTITUÉ : {fichier_corpus}')
print(f'📊 Volume total : {len(lignes_dialogues):,} dialogues & échanges ({fichier_corpus.stat().st_size / 1e6:.2f} Mo)')
print('='*70)

## 4. Entraînement GPU VRAM Ultime (10 Go+ VRAM & Débit > 10M-30M Tok/s)

Cette cellule active l'allocation VRAM maximale (cible : 10.0 Go sur les 15 Go du GPU), lance les 3 CUDA Streams parallèles et gonfle `ratiss_self/bouche.bouche` (22,9 Mo existants) en continu !

In [ ]:
import os, shutil
from pathlib import Path

SORTIE_DIR = 'sortie/langue_massive'
MODELE_REPRISE = 'ratiss_self/bouche.bouche'

cmd = [
    'python3', 'entrainement/gpu/entrainer_gpu.py',
    '--corpus', 'corpus/corpus_colossal.txt',
    '--sortie', SORTIE_DIR,
    '--registre', 'dialogue',
    '--tours', '2',
    '--fraction', '0.05',
    '--vram-cible', '10.0'  # Exploitation maximale de la VRAM (10 Go sur 15 Go) !
]

if os.path.exists(MODELE_REPRISE):
    taille_mo = os.path.getsize(MODELE_REPRISE) / 1e6
    print(f'🔄 REPRISE ET GONFLAGE DU MODÈLE : {MODELE_REPRISE} ({taille_mo:.2f} Mo / 881k arcs) !')
    cmd += ['--charger', MODELE_REPRISE]
else:
    print('🌱 Initialisation vierge.')

# Exécution de l'entraînement GPU VRAM Ultime
!{' '.join(cmd)}

# Persistance automatique et immédiate dans ratiss_self/
source_bouche = Path(f'{SORTIE_DIR}/bouche.bouche')
if source_bouche.exists():
    dest_self = Path('ratiss_self/bouche.bouche')
    dest_self.parent.mkdir(exist_ok=True)
    shutil.copy2(source_bouche, dest_self)
    taille_finale_mo = dest_self.stat().st_size / 1e6
    print(f'\n💾 PERSISTANCE VALIDÉE : Modèle scellé colossale sauvegardé dans {dest_self} ({taille_finale_mo:.2f} Mo) !')

print('\n' + '='*50)
print(open(f'{SORTIE_DIR}/ORGANISATION.txt', encoding='utf-8').read())

## 5. Banc d'Essai de Parole & Dialogue en Direct (6 Tests Conversationnels)

Testons directement la capacité conversationnelle de la Bouche ultra-gonflée sur différentes questions et répliques !

In [ ]:
import subprocess

TESTS_CONVERSATIONNELS = [
    ('1. Identité & Conscience', '_q_ qui es tu et quelle est ta nature _r_', 0.75, 8, True),
    ('2. Question Scientifique (Soleil & Lune)', '_q_ parle moi du soleil et de la lune _r_', 0.65, 8, True),
    ('3. Connaissance Fondamentale (Gravité)', '_q_ comment fonctionne la gravite _r_', 0.75, 8, True),
    ('4. Aventure & Exploration Maritime', '_q_ comment explorer les profondeurs de l ocean _r_', 0.75, 8, True),
    ('5. Échange Émotionnel & Esprit', '_q_ que penses tu du courage et de l honneur _r_', 0.70, 8, True),
    ('6. Mystère du Monde (Génération Libre)', '_q_ raconte moi un secret sur le monde _r_', 0.75, 8, False),
]

print('='*70)
print('🎙️ BANC D\'ESSAI CONVERSATIONNEL DU MODÈLE ULTRA-GONFLÉ')
print('='*70)

for titre, amorce, lam, mem, glouton in TESTS_CONVERSATIONNELS:
    print(f'\n👉 {titre}')
    print(f'   Question / Amorce : "{amorce}"')
    cmd = [
        'python3', 'ratiss_v3.py', 'cerveau/RatissOne.ratiss',
        '--charger', 'ratiss_self/bouche.bouche',
        '--lam', str(lam),
        '--memoire', str(mem),
        '--mots', '12',
        amorce
    ]
    if glouton:
        cmd.append('--glouton')
    res = subprocess.run(cmd, capture_output=True, text=True)
    suite = res.stdout.strip().splitlines()[-1] if res.stdout.strip() else '...'
    print(f'   Réponse Bouche   : \033[92m{suite}\033[0m')

## 6. Sauvegarde Directe & Téléchargement Chrome (Zéro Clé Requise)

Cette cellule compresse automatiquement tous les artefacts d'entraînement (`bouche.bouche`, `mesures.json`, `ORGANISATION.txt`, etc.) et déclenche le téléchargement direct sur votre ordinateur via votre navigateur Chrome.

In [ ]:
from google.colab import files
import shutil, os
from pathlib import Path

print('='*60)
print('📥 SAUVEGARDE DIRECTE ET TÉLÉCHARGEMENT CHROME')
print('='*60)

dossier_export = Path('sortie/ratiss_colossal')
dossier_export.mkdir(parents=True, exist_ok=True)

# Rassemblement des artefacts du modèle colossal
fichiers = [
    'ratiss_self/bouche.bouche',
    'sortie/langue_massive/bouche.bouche',
    'sortie/langue_massive/mesures.json',
    'sortie/langue_massive/ORGANISATION.txt',
    'sortie/langue_massive/01_education.txt',
    'sortie/langue_massive/02_ecart.txt'
]

for f in fichiers:
    if os.path.exists(f):
        shutil.copy2(f, dossier_export)
        print(f'  ✓ Préparé : {f}')

# Création de l\'archive zip officielle
archive_nom = 'ratiss_bouche_colossale_gonflee'
shutil.make_archive(archive_nom, 'zip', dossier_export)
archive_zip = f'{archive_nom}.zip'

taille_zip_mo = os.path.getsize(archive_zip) / 1e6
print(f'\n📦 Archive prête : {archive_zip} ({taille_zip_mo:.2f} Mo)')
print('🚀 Déclenchement automatique du téléchargement Chrome...')

# Téléchargement direct vers le navigateur
files.download(archive_zip)
files.download('ratiss_self/bouche.bouche')

print('🎉 Téléchargement Chrome déclenché avec succès !')